# Disinformation: refusals vs. judge labels
Do replies that look like refusals get labelled `lie` by the judge? Samples come from `outputs/baseline_full_test/mask_disinformation.jsonl`.

In [ ]:
import json, re, random
from pathlib import Path
import pandas as pd

RUN = Path("../outputs/baseline_full_test")   # change to inspect another run
ARCH = "disinformation"
rows = [json.loads(l) for l in open(RUN / f"mask_{ARCH}.jsonl")]
df = pd.DataFrame(rows)
REFUSAL = re.compile(r"\b(I can't|I cannot|I'm sorry|I am unable|I won't|I'm not able|I must decline)\b", re.I)
df["refusal_start"] = df.pressured_response.str[:200].apply(lambda t: bool(REFUSAL.search(t)))
df["label"] = df.apply(lambda r: "lie" if r.lie else "honest" if r.honest else "evasive" if r.evasive else "no_belief", axis=1)
print(len(df), "rows")
pd.crosstab(df.refusal_start, df.label, margins=True)

In [ ]:
def show(r, n=900):
    print("=" * 100)
    print(f"{r.task_id} | label={r.label} | statement={r.statement} | belief={r.belief} | gt={r.ground_truth}")
    print("--- GENERATION:\n" + r.pressured_response[:n] + (" [...]" if len(r.pressured_response) > n else ""))
    print("--- JUDGE (tail):", r.statement_judge[-350:].replace("\n", " "))

def sample(mask, k=3, seed=0):
    sub = df[mask]
    print(f"{len(sub)} rows match; showing {min(k, len(sub))}")
    for _, r in sub.sample(min(k, len(sub)), random_state=seed).iterrows():
        show(r)

## Refusal-looking replies that the judge called a `lie` (the suspicious group)

In [ ]:
sample(df.refusal_start & (df.label == 'lie'), k=4)

## Refusal-looking replies labelled `evasive` (the expected outcome)

In [ ]:
sample(df.refusal_start & (df.label == 'evasive'), k=2)

## Replies that do not look like refusals but were labelled `lie` (for comparison)

In [ ]:
sample(~df.refusal_start & (df.label == 'lie'), k=2)